### Preamble

In [5]:
import sumolib
from pathlib import Path
import subprocess
import xml.etree.ElementTree as ET

from app.network.road import build_lane_records, build_junction_records, compute_bounds, compute_edge_markings, compute_lane_markings
from app.network.opposite_marking import compute_opposite_direction_markings
from app.network.util import SVG

%load_ext autoreload
%autoreload 2

output_dir = "spread-type"
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Comparison of `spreadType`

In [6]:
nodes_file = output_dir / f"network.nod.xml"
edges_file = output_dir / f"network.edg.xml"
net_file = output_dir / f"network.net.xml"

import xml.etree.ElementTree as ET
nodes_root = ET.Element("nodes")
ET.SubElement(nodes_root, "node", id="node_1", type="priority", x="0.00", y="0.00")
ET.SubElement(nodes_root, "node", id="node_2", type="priority", x="50.00", y="0.00")

ET.SubElement(nodes_root, "node", id="node_3", type="priority", x="0.00", y="12.00")
ET.SubElement(nodes_root, "node", id="node_4", type="priority", x="50.00", y="12.00")

ET.SubElement(nodes_root, "node", id="node_5", type="priority", x="0.00", y="24.00")
ET.SubElement(nodes_root, "node", id="node_6", type="priority", x="50.00", y="24.00")

ET.SubElement(nodes_root, "node", id="node_7", type="priority", x="0.00", y="36.00")
ET.SubElement(nodes_root, "node", id="node_8", type="priority", x="50.00", y="36.00")

ET.indent(nodes_root, space="  ")
ET.ElementTree(nodes_root).write(nodes_file)

edges_root = ET.Element("edges")
ET.SubElement(edges_root, "edge", id="edge_1", width="2.00",
        **{ "from": "node_1", "to": "node_2", "numLanes": "2", "spreadType": "roadCenter" }
)
ET.SubElement(edges_root, "edge", id="edge_2", width="2.00",
        **{ "from": "node_2", "to": "node_1", "numLanes": "3", "spreadType": "roadCenter" }
)

ET.SubElement(edges_root, "edge", id="edge_3", width="2.00",
        **{ "from": "node_3", "to": "node_4", "numLanes": "2", "spreadType": "right" }
)
ET.SubElement(edges_root, "edge", id="edge_4", width="2.00",
        **{ "from": "node_4", "to": "node_3", "numLanes": "3", "spreadType": "right" }
)

ET.SubElement(edges_root, "edge", id="edge_5", width="2.00",
        **{ "from": "node_5", "to": "node_6", "numLanes": "3", "spreadType": "center" }
)

ET.SubElement(edges_root, "edge", id="edge_6", width="2.00",
        **{ "from": "node_7", "to": "node_8", "numLanes": "3", "spreadType": "right" }
)

ET.indent(edges_root, space="  ")
ET.ElementTree(edges_root).write(edges_file)

netconvert_bin = sumolib.checkBinary("netconvert")
subprocess.run([
    netconvert_bin,
    "--node-files", str(nodes_file),
    "--edge-files", str(edges_file),
    "--output-file", str(net_file),
], check=True)

tree = ET.parse(net_file)
root = tree.getroot()

Success.


In [7]:
# Construct lane records
lane_records = build_lane_records(root)
junction_records = build_junction_records(root)

# Collect the road polygons
lane_polys = [rec["polygon"] for rec in lane_records]
junc_polys = [rec["polygon"] for rec in junction_records]

# Compute bounds for viewport fitting
bounds = compute_bounds(lane_polys + junc_polys)

# Compute lane markings
lane_markings = compute_lane_markings(lane_records)
edge_markings = compute_edge_markings(lane_polys + junc_polys)
opposite_markings, opposite_marking_debug = compute_opposite_direction_markings(lane_records, opposite_seam_style="solid")

In [8]:
bounds = {'minx': -20, 'miny': -38.0, 'maxx': 56.0, 'maxy': 7.0}
svg = SVG(bounds)

svg.draw_polygons(lane_polys, fill="#aaa", stroke="#aaa", stroke_width=0.2)
svg.draw_polygons(junc_polys, fill="#aaa", stroke="#aaa", stroke_width=0.2)

# svg.draw_polygons(edge_markings, stroke="red", stroke_width=0.1)
svg.draw_polygons(lane_markings, stroke="white", fill="white", stroke_width=0.2)
svg.draw_polygons(opposite_markings, stroke="white", fill="white", stroke_width=0.2)

# draw the edge centerlines
svg.draw_points([(0, 0), (50, 0)], r=0.3)
svg.draw_lines([(0, 0, 50, 0)], stroke_width=0.1, dashed=True)
svg.draw_text(-10, 0, "roadCenter", font_size=2, text_anchor="middle")

svg.draw_points([(0, -12), (50, -12)], r=0.3)
svg.draw_lines([(0, -12, 50, -12)], stroke_width=0.1, dashed=True)
svg.draw_text(-10, -12, "right/center", font_size=2, text_anchor="middle")

svg.draw_points([(0, -24), (50, -24)], r=0.3)
svg.draw_lines([(0, -24, 50, -24)], stroke_width=0.1, dashed=True)
svg.draw_text(-10, -24, "center", font_size=2, text_anchor="middle")

svg.draw_points([(0, -36), (50, -36)], r=0.3)
svg.draw_lines([(0, -36, 50, -36)], stroke_width=0.1, dashed=True)
svg.draw_text(-10, -33, "right/roadCenter", font_size=2, text_anchor="middle")

svg.write(output_dir / "network.svg")